<a href="https://colab.research.google.com/github/sayyadaliya90-sketch/Air-Quality-Index/blob/main/MLOpsTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.makedirs("mlops-task", exist_ok=True)

print("Project Folder Created")

Project Folder Created


In [2]:
config = """
seed: 42
window: 5
version: "v1"
"""

with open("mlops-task/config.yaml","w") as f:
    f.write(config)

print("config.yaml created")

config.yaml created


In [3]:
!cat mlops-task/config.yaml


seed: 42
window: 5
version: "v1"


In [4]:
from google.colab import files

uploaded = files.upload()


Saving data.csv - Sheet1.csv to data.csv - Sheet1.csv


In [7]:
!ls


'data.csv - Sheet1.csv'   mlops-task   sample_data


In [8]:
import shutil

shutil.move("data.csv - Sheet1.csv", "mlops-task/data.csv")

print("File Moved")

File Moved


In [9]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print(df.head())
print(df.columns)

  timestamp,open,high,low,close,volume_btc,volume_usd
0  2024-01-01 00:00:00,44910.83,45085.78,44910.83... 
1  2024-01-01 00:01:00,44977.19,45045.33,44977.19... 
2  2024-01-01 00:02:00,44991.38,45103.85,44965.65... 
3  2024-01-01 00:03:00,45056.78,45135.78,45056.78... 
4  2024-01-01 00:04:00,45172.02,45222.19,44979.72... 
Index(['timestamp,open,high,low,close,volume_btc,volume_usd'], dtype='object')


In [10]:
print(df.columns)


Index(['timestamp,open,high,low,close,volume_btc,volume_usd'], dtype='object')


In [11]:
with open("mlops-task/data.csv", "r") as f:
    for i in range(5):
        print(repr(f.readline()))

'"timestamp,open,high,low,close,volume_btc,volume_usd"\n'
'"2024-01-01 00:00:00,44910.83,45085.78,44910.83,45024.68,3.640837,163927.55"\n'
'"2024-01-01 00:01:00,44977.19,45045.33,44977.19,45017.83,33.752474,1519463.07"\n'
'"2024-01-01 00:02:00,44991.38,45103.85,44965.65,45050.03,5.488855,247273.08"\n'
'"2024-01-01 00:03:00,45056.78,45135.78,45056.78,45125.82,8.464271,381957.14"\n'


In [12]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv", sep=",")

print(df.head())
print(df.columns.tolist())

  timestamp,open,high,low,close,volume_btc,volume_usd
0  2024-01-01 00:00:00,44910.83,45085.78,44910.83... 
1  2024-01-01 00:01:00,44977.19,45045.33,44977.19... 
2  2024-01-01 00:02:00,44991.38,45103.85,44965.65... 
3  2024-01-01 00:03:00,45056.78,45135.78,45056.78... 
4  2024-01-01 00:04:00,45172.02,45222.19,44979.72... 
['timestamp,open,high,low,close,volume_btc,volume_usd']


In [14]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 10000
Columns: ['timestamp,open,high,low,close,volume_btc,volume_usd']


In [15]:
requirements = """
pandas
numpy
pyyaml
"""

with open("mlops-task/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [16]:
!pip install pandas numpy pyyaml

In [18]:
run_code = """
print("Hello")
"""

In [19]:
with open("mlops-task/run.py", "w") as f:
    f.write(run_code)

print("run.py created")

run.py created


In [20]:
%%writefile mlops-task/run.py

print("Hello")

Overwriting mlops-task/run.py


In [21]:
!ls mlops-task

config.yaml  data.csv  requirements.txt  run.py


In [22]:
%%writefile mlops-task/run.py

import argparse
import pandas as pd
import numpy as np
import yaml
import json
import time
import logging

parser = argparse.ArgumentParser()

parser.add_argument("--input", required=True)
parser.add_argument("--config", required=True)
parser.add_argument("--output", required=True)
parser.add_argument("--log-file", required=True)

args = parser.parse_args()

logging.basicConfig(
    filename=args.log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

start_time = time.time()

try:
    logging.info("Job started")

    with open(args.config, "r") as f:
        config = yaml.safe_load(f)

    required = ["seed", "window", "version"]

    for key in required:
        if key not in config:
            raise ValueError(f"Missing config field: {key}")

    seed = config["seed"]
    window = config["window"]
    version = config["version"]

    np.random.seed(seed)

    df = pd.read_csv(args.input)

    if df.empty:
        raise ValueError("CSV file is empty")

    if "close" not in df.columns:
        raise ValueError("close column missing")

    df["rolling_mean"] = df["close"].rolling(window=window).mean()

    df["signal"] = (df["close"] > df["rolling_mean"]).astype(int)

    latency_ms = int((time.time() - start_time) * 1000)

    metrics = {
        "version": version,
        "rows_processed": len(df),
        "metric": "signal_rate",
        "value": round(df["signal"].mean(), 4),
        "latency_ms": latency_ms,
        "seed": seed,
        "status": "success"
    }

    with open(args.output, "w") as f:
        json.dump(metrics, f, indent=4)

    logging.info("Job completed")

    print(json.dumps(metrics, indent=4))

except Exception as e:

    error_output = {
        "version": "v1",
        "status": "error",
        "error_message": str(e)
    }

    with open(args.output, "w") as f:
        json.dump(error_output, f, indent=4)

    logging.exception("Job failed")

    print(json.dumps(error_output, indent=4))

Overwriting mlops-task/run.py


In [23]:
!python mlops-task/run.py \
--input mlops-task/data.csv \
--config mlops-task/config.yaml \
--output mlops-task/metrics.json \
--log-file mlops-task/run.log

{
    "version": "v1",
    "status": "error",
    "error_message": "close column missing"
}


In [24]:
!cat mlops-task/metrics.json

{
    "version": "v1",
    "status": "error",
    "error_message": "close column missing"
}

In [25]:
!cat mlops-task/run.log

2026-06-08 09:56:08,395 - INFO - Job started
2026-06-08 09:56:08,425 - ERROR - Job failed
Traceback (most recent call last):
  File "/content/mlops-task/run.py", line 51, in <module>
    raise ValueError("close column missing")
ValueError: close column missing


In [26]:
with open("mlops-task/data.csv", "r") as f:
    for i in range(5):
        print(repr(f.readline()))

'"timestamp,open,high,low,close,volume_btc,volume_usd"\n'
'"2024-01-01 00:00:00,44910.83,45085.78,44910.83,45024.68,3.640837,163927.55"\n'
'"2024-01-01 00:01:00,44977.19,45045.33,44977.19,45017.83,33.752474,1519463.07"\n'
'"2024-01-01 00:02:00,44991.38,45103.85,44965.65,45050.03,5.488855,247273.08"\n'
'"2024-01-01 00:03:00,45056.78,45135.78,45056.78,45125.82,8.464271,381957.14"\n'


In [27]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print(df.columns.tolist())

['timestamp,open,high,low,close,volume_btc,volume_usd']


In [28]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv", sep=";")

print(df.columns.tolist())
df.head()

['timestamp,open,high,low,close,volume_btc,volume_usd']


,"timestamp,open,high,low,close,volume_btc,volume_usd"
0,"2024-01-01 00:00:00,44910.83,45085.78,44910.83..."
1,"2024-01-01 00:01:00,44977.19,45045.33,44977.19..."
2,"2024-01-01 00:02:00,44991.38,45103.85,44965.65..."
3,"2024-01-01 00:03:00,45056.78,45135.78,45056.78..."
4,"2024-01-01 00:04:00,45172.02,45222.19,44979.72..."


In [30]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print(df.columns.tolist())

['timestamp,open,high,low,close,volume_btc,volume_usd']


In [31]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print(df.columns.tolist())

['timestamp,open,high,low,close,volume_btc,volume_usd']


In [33]:
with open("mlops-task/data.csv","r") as f:
    print(f.readline())

"timestamp,open,high,low,close,volume_btc,volume_usd"



In [34]:
import pandas as pd

df = pd.read_csv("mlops-task/data.csv")

print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Columns:
['timestamp,open,high,low,close,volume_btc,volume_usd']

First 5 rows:
  timestamp,open,high,low,close,volume_btc,volume_usd
0  2024-01-01 00:00:00,44910.83,45085.78,44910.83... 
1  2024-01-01 00:01:00,44977.19,45045.33,44977.19... 
2  2024-01-01 00:02:00,44991.38,45103.85,44965.65... 
3  2024-01-01 00:03:00,45056.78,45135.78,45056.78... 
4  2024-01-01 00:04:00,45172.02,45222.19,44979.72... 


In [35]:
import re

with open("mlops-task/run.py", "r") as f:
    code = f.read()

code = code.replace(
    'df = pd.read_csv(args.input)',
    'df = pd.read_csv(args.input, sep=None, engine="python")'
)

with open("mlops-task/run.py", "w") as f:
    f.write(code)

print("run.py updated")

run.py updated


In [36]:
!python mlops-task/run.py \
--input mlops-task/data.csv \
--config mlops-task/config.yaml \
--output mlops-task/metrics.json \
--log-file mlops-task/run.log

{
    "version": "v1",
    "status": "error",
    "error_message": "close column missing"
}


In [37]:
!cat mlops-task/metrics.json

{
    "version": "v1",
    "status": "error",
    "error_message": "close column missing"
}

In [38]:
!ls mlops-task

config.yaml  data.csv  metrics.json  requirements.txt  run.log	run.py


In [39]:
docker_content = """
FROM python:3.9-slim

WORKDIR /app

COPY . .

RUN pip install -r requirements.txt

CMD ["python",
     "run.py",
     "--input","data.csv",
     "--config","config.yaml",
     "--output","metrics.json",
     "--log-file","run.log"]
"""

with open("mlops-task/Dockerfile", "w") as f:
    f.write(docker_content)

print("Dockerfile created")

Dockerfile created


In [40]:
readme_content = """
# MLOps Assignment

## Project Structure

- run.py
- config.yaml
- data.csv
- requirements.txt
- Dockerfile
- metrics.json
- run.log

## Install Dependencies

pip install -r requirements.txt

## Run

python run.py --input data.csv --config config.yaml --output metrics.json --log-file run.log

## Docker Build

docker build -t mlops-task .

## Docker Run

docker run --rm mlops-task

## Output

The script generates:
- metrics.json
- run.log

## Metric

signal_rate = mean(signal)

signal = 1 if close > rolling_mean else 0
"""
with open("mlops-task/README.md", "w") as f:
    f.write(readme_content)

print("README.md created")

README.md created


In [41]:
!ls mlops-task

config.yaml  Dockerfile    README.md	     run.log
data.csv     metrics.json  requirements.txt  run.py


In [42]:
!zip -r mlops-task.zip mlops-task

  adding: mlops-task/ (stored 0%)
  adding: mlops-task/requirements.txt (stored 0%)
  adding: mlops-task/config.yaml (stored 0%)
  adding: mlops-task/run.py (deflated 61%)
  adding: mlops-task/Dockerfile (deflated 29%)
  adding: mlops-task/data.csv (deflated 64%)
  adding: mlops-task/README.md (deflated 46%)
  adding: mlops-task/metrics.json (deflated 21%)
  adding: mlops-task/run.log (deflated 60%)


In [43]:
from google.colab import files
files.download("mlops-task.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>